# 投稿採点モデル — ファインチューニング

BERTをベースに、SNS投稿を採点するモデルを作る。

**入力**: 投稿テキスト  
**出力**:
- `hook` : フックの強さ (1〜5)
- `specificity` : 具体性 (1〜5)
- `clarity` : 明確さ・一貫性 (1〜5)
- `relatability` : 共感・自分ごと化 (1〜5)
- `binary` : good(1) / bad(0)

採点基準は `scoring_rubric.md` を参照。

**実行手順**
1. `labels.jsonl` をColabにアップロード（`{"text","hook","specificity","clarity","relatability","binary"}`）
2. 上から順にセルを実行
3. 最後にモデルが `post_evaluator/` に保存される

## 0. ライブラリのインストール

In [4]:
!pip install transformers torch scikit-learn -q

## 1. データの読み込みと確認

In [5]:
# Google Drive をマウントしてパスを設定する
from google.colab import drive
drive.mount('/content/drive')

import os

# Drive上の作業フォルダ(なければ自動作成)
DRIVE_DIR = '/content/drive/MyDrive/post_evaluator_project'
os.makedirs(DRIVE_DIR, exist_ok=True)

# 各ファイルのパス
DATA_PATH  = f'{DRIVE_DIR}/labels.jsonl'   # 学習データ
SAVE_DIR   = f'{DRIVE_DIR}/post_evaluator' # モデル保存先
BEST_MODEL = f'{DRIVE_DIR}/best_model.pt'  # 学習中の一時保存

print(f'Drive接続完了')
print(f'作業フォルダ: {DRIVE_DIR}')
print(f'データ: {DATA_PATH}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive接続完了
作業フォルダ: /content/drive/MyDrive/post_evaluator_project
データ: /content/drive/MyDrive/post_evaluator_project/labels.jsonl


In [ ]:
import json
import pandas as pd

# labels.jsonl を読み込む
records = []
with open(DATA_PATH, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line.strip()))

df = pd.DataFrame(records)
print(f'データ件数: {len(df)}')
print(f'カラム: {df.columns.tolist()}')
print()
print('--- 採点の分布 ---')
print(f'binary  good:{(df.binary==1).sum()}件  bad:{(df.binary==0).sum()}件')
for col in ['hook', 'specificity', 'clarity', 'relatability']:
    print(f'{col:12} 平均:{df[col].mean():.2f}  min:{df[col].min()}  max:{df[col].max()}')
df.head(3)

## 2. データセットの準備

BERTに渡すために、テキストをトークン(数値の列)に変換する。

**トークナイザーとは**: 文章を単語・サブワード単位に分割して、BERTが理解できる数値列に変換するもの。

例: `「起動0.5秒」→ [101, 5823, 102, ...]`

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split

# 変更後(Driveにあればそこから、なければHugging Faceから)
MODEL_NAME    = 'cl-tohoku/bert-base-japanese-v3'
BERT_SAVE_DIR = f'{DRIVE_DIR}/bert_base_japanese'

if os.path.exists(BERT_SAVE_DIR):
    print(f'Driveからトークナイザーを読み込みます: {BERT_SAVE_DIR}')
    tokenizer = BertTokenizer.from_pretrained(BERT_SAVE_DIR)
else:
    print(f'Hugging Faceからダウンロードします(初回のみ)')
    tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

# 動作確認
sample = '起動0.5秒で開くタスク管理アプリ'
tokens = tokenizer(sample, return_tensors='pt')
print(f'サンプルテキスト: {sample}')
print(f'トークンID: {tokens["input_ids"][0].tolist()}')
print(f'トークン数: {len(tokens["input_ids"][0])}')


# BERTの事前学習済みモデルをDriveに保存する
# 初回だけ実行。次回以降はDriveから読み込むので不要。

BERT_SAVE_DIR = f'{DRIVE_DIR}/bert_base_japanese'

if os.path.exists(BERT_SAVE_DIR):
    print(f'Driveにすでに保存済みです: {BERT_SAVE_DIR}')
    print('スキップします')
else:
    from transformers import BertModel
    print('BERTの事前学習済み重みをDriveに保存中...')

    # モデル本体を保存
    bert_model = BertModel.from_pretrained(MODEL_NAME)
    bert_model.save_pretrained(BERT_SAVE_DIR)

    # トークナイザーも同じフォルダに保存(セット管理のため)
    tokenizer.save_pretrained(BERT_SAVE_DIR)

    print(f'保存完了: {BERT_SAVE_DIR}')
    for fname in os.listdir(BERT_SAVE_DIR):
        size = os.path.getsize(f'{BERT_SAVE_DIR}/{fname}') / 1024 / 1024
        print(f'  {fname}: {size:.1f} MB')

Driveからトークナイザーを読み込みます: /content/drive/MyDrive/post_evaluator_project/bert_base_japanese
サンプルテキスト: 起動0.5秒で開くタスク管理アプリ
トークンID: [2, 5829, 1083, 31, 29, 36, 4267, 457, 6349, 433, 21990, 7096, 4426, 3822, 15041, 3]
トークン数: 16
Driveにすでに保存済みです: /content/drive/MyDrive/post_evaluator_project/bert_base_japanese
スキップします


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split


def _norm(v):
    """1〜5 を 0〜1 に正規化（回帰ヘッドがsigmoidで0〜1を出すため）。"""
    return torch.tensor((v - 1) / 4, dtype=torch.float)


class PostDataset(Dataset):
    """BERTに渡すデータセット。4つの回帰ラベル(0〜1) + 1つの分類ラベルを返す。"""
    def __init__(self, frame, tokenizer, max_len=128):
        self.texts = frame['text'].tolist()
        self.hooks = frame['hook'].tolist()
        self.specificities = frame['specificity'].tolist()
        self.clarities = frame['clarity'].tolist()
        self.relatabilities = frame['relatability'].tolist()
        self.binaries = frame['binary'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'hook':           _norm(self.hooks[idx]),
            'specificity':    _norm(self.specificities[idx]),
            'clarity':        _norm(self.clarities[idx]),
            'relatability':   _norm(self.relatabilities[idx]),
            'binary':         torch.tensor(self.binaries[idx], dtype=torch.long),
        }


# 訓練データ・検証データに分割 (8:2)
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['binary'])
print(f'訓練データ: {len(train_df)}件  検証データ: {len(val_df)}件')


def make_loader(frame, shuffle=True):
    return DataLoader(PostDataset(frame, tokenizer), batch_size=8, shuffle=shuffle)


train_loader = make_loader(train_df, shuffle=True)
val_loader   = make_loader(val_df,   shuffle=False)
print('DataLoader作成完了')

## 3. モデルの定義

BERTの上に5つのヘッドを乗せる。

```
テキスト
  ↓ トークン化
BERT (事前学習済み)
  ↓ [CLS]トークンのベクトル(768次元)
  ├─ hook_head         → フックスコア(0〜1)
  ├─ specificity_head  → 具体性スコア(0〜1)
  ├─ clarity_head      → 明確さスコア(0〜1)
  ├─ relatability_head → 共感スコア(0〜1)
  └─ binary_head       → good/bad(2クラス)
```

各ヘッドの属性名は推論側 `local_evaluator.py` の `PostEvaluatorModel` と一致させること
（state_dictのキーになるため）。

In [ ]:
import torch.nn as nn
from transformers import BertModel

class PostEvaluator(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()

        # DriveにあればそこからBERTを読み込む(ダウンロード不要)
        bert_source = BERT_SAVE_DIR if os.path.exists(BERT_SAVE_DIR) else MODEL_NAME
        print(f'BERTの読み込み元: {bert_source}')
        self.bert = BertModel.from_pretrained(bert_source)

        self.dropout           = nn.Dropout(dropout)
        self.hook_head         = nn.Linear(768, 1)
        self.specificity_head  = nn.Linear(768, 1)
        self.clarity_head      = nn.Linear(768, 1)
        self.relatability_head = nn.Linear(768, 1)
        self.binary_head       = nn.Linear(768, 2)
        self.sigmoid           = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        outputs    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_vector = outputs.last_hidden_state[:, 0, :]
        cls_vector = self.dropout(cls_vector)
        hook         = self.sigmoid(self.hook_head(cls_vector)).squeeze(-1)
        specificity  = self.sigmoid(self.specificity_head(cls_vector)).squeeze(-1)
        clarity      = self.sigmoid(self.clarity_head(cls_vector)).squeeze(-1)
        relatability = self.sigmoid(self.relatability_head(cls_vector)).squeeze(-1)
        binary       = self.binary_head(cls_vector)
        return hook, specificity, clarity, relatability, binary

# GPUが使えるか確認
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'使用デバイス: {device}')

model = PostEvaluator().to(device)
print(f'パラメータ数: {sum(p.numel() for p in model.parameters()):,}')

## 4. 学習

損失関数は3つの組み合わせ:
- `MSELoss`: フック・具体性の回帰誤差(予測値と正解の差の二乗)
- `CrossEntropyLoss`: good/bad分類の誤差

最適化アルゴリズムはAdamW(BERTのファインチューニングの定番)。

In [ ]:
from torch.optim import AdamW

# 損失関数
mse  = nn.MSELoss()             # 回帰用(hook/specificity/clarity/relatability)
ce   = nn.CrossEntropyLoss()    # 分類用(binary)

# 最適化アルゴリズム。lr=2e-5 はBERTファインチューニングの定番値
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

def run_epoch(loader, is_train=True):
    """1エポック分の学習または評価を実行して、平均損失を返す。"""
    model.train(is_train)
    total_loss = 0

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        y_hook = batch['hook'].to(device)
        y_spec = batch['specificity'].to(device)
        y_clar = batch['clarity'].to(device)
        y_rel  = batch['relatability'].to(device)
        y_bin  = batch['binary'].to(device)

        with torch.set_grad_enabled(is_train):
            p_hook, p_spec, p_clar, p_rel, p_bin = model(input_ids, attention_mask)

            # 回帰4つ + 分類1つ。回帰は分類とスケールを揃えるため重み2.0
            loss = (
                mse(p_hook, y_hook) * 2.0
              + mse(p_spec, y_spec) * 2.0
              + mse(p_clar, y_clar) * 2.0
              + mse(p_rel,  y_rel)  * 2.0
              + ce(p_bin, y_bin)    * 1.0
            )

        if is_train:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

# 学習ループ
EPOCHS = 10
best_val_loss = float('inf')

print('学習開始')
print(f'{"Epoch":>6} {"Train Loss":>12} {"Val Loss":>10}')
print('-' * 32)

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, is_train=True)
    val_loss   = run_epoch(val_loader,   is_train=False)

    marker = ' ← best' if val_loss < best_val_loss else ''
    print(f'{epoch:>6} {train_loss:>12.4f} {val_loss:>10.4f}{marker}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), BEST_MODEL)

print(f'\n学習完了。最良の検証損失: {best_val_loss:.4f}')

## 5. 評価 — 検証データで精度を確認

In [ ]:
from sklearn.metrics import accuracy_score, mean_absolute_error
import numpy as np

# 最良モデルを読み込む
model.load_state_dict(torch.load(BEST_MODEL, map_location=device))
model.eval()

axes = ['hook', 'specificity', 'clarity', 'relatability']
pred = {a: [] for a in axes}
true = {a: [] for a in axes}
bin_pred, bin_true = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outs = model(input_ids, attention_mask)   # hook, spec, clarity, relatability, binary
        for a, out in zip(axes, outs[:4]):
            # 0〜1 → 1〜5に戻す
            pred[a].extend((out.cpu().numpy() * 4 + 1).round().clip(1, 5).astype(int))
            true[a].extend((batch[a].numpy() * 4 + 1).round().astype(int))
        bin_pred.extend(outs[4].argmax(dim=1).cpu().numpy())
        bin_true.extend(batch['binary'].numpy())

print('=== 検証データでの評価結果 ===')
for a in axes:
    print(f'{a:12} MAE: {mean_absolute_error(true[a], pred[a]):.3f} (1〜5スケール)')
print(f'good/bad 正解率 : {accuracy_score(bin_true, bin_pred):.3f}')
print()
print('--- 予測の内訳(先頭5件) ---')
for text, h, s, c, r, b in zip(
    val_df['text'].tolist()[:5],
    pred['hook'][:5], pred['specificity'][:5], pred['clarity'][:5],
    pred['relatability'][:5], bin_pred[:5]
):
    print(f'投稿: {text[:30]}...')
    print(f'  予測: フック{h} 具体性{s} 明確さ{c} 共感{r} / {"good" if b==1 else "bad"}')
    print()

## 6. モデルの保存

次回から再学習不要で使えるように、モデルの重みと設定をまとめて保存する。

In [ ]:
import os, json

os.makedirs(SAVE_DIR, exist_ok=True)

# モデルの重みを保存（推論側 local_evaluator.py がこの pytorch_model.pt を読む）
torch.save(model.state_dict(), f'{SAVE_DIR}/pytorch_model.pt')

# トークナイザーを保存
tokenizer.save_pretrained(SAVE_DIR)

# メタ情報を保存
meta = {
    'base_model': MODEL_NAME,
    'max_len': 128,
    'axes': ['hook', 'specificity', 'clarity', 'relatability'],
    'best_val_loss': best_val_loss,
    'train_size': len(train_df),
    'val_size': len(val_df),
}
with open(f'{SAVE_DIR}/meta.json', 'w') as f:
    json.dump(meta, f, indent=2, ensure_ascii=False)

print(f'Driveに保存しました: {SAVE_DIR}')
for fname in os.listdir(SAVE_DIR):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}') / 1024 / 1024
    print(f'  {fname}: {size:.1f} MB')

## 7. 推論テスト — 任意のテキストを採点してみる

In [ ]:
def predict(text, model, tokenizer, device):
    """テキストを採点して結果を返す。"""
    model.eval()
    inputs = tokenizer(
        text, max_length=128, padding='max_length', truncation=True, return_tensors='pt'
    )
    with torch.no_grad():
        p_hook, p_spec, p_clar, p_rel, p_bin = model(
            inputs['input_ids'].to(device),
            inputs['attention_mask'].to(device)
        )

    def to15(x):
        return max(1, min(5, int(x.item() * 4 + 1)))

    return {
        'hook':         to15(p_hook),
        'specificity':  to15(p_spec),
        'clarity':      to15(p_clar),
        'relatability': to15(p_rel),
        'binary':       'good' if p_bin.argmax().item() == 1 else 'bad',
    }

# テスト投稿で確認
test_posts = [
    # 高スコアを期待（フック・具体・共感あり）
    '「なんでこんなに重いんだ」と思いながら使っていたタスク管理ツール、自分で作ったら起動0.5秒になった。削ぎ落とすって、足すより難しい。#個人開発',
    # 低スコアを期待（宣伝・抽象）
    'FocusFlowというアプリを作りました。タスク管理に特化したシンプルなアプリです。ぜひ使ってみてください。',
    # 明確さ最低を期待（メタ文・散らかり）
    'はい、承知いたしました。以下に投稿案を作成します。色々な機能があって便利で、ぜひ使ってほしいです。',
]

print('=== 推論テスト ===')
for post in test_posts:
    r = predict(post, model, tokenizer, device)
    total = r['hook'] + r['specificity'] + r['clarity'] + r['relatability']
    print(f'投稿: {post[:40]}...')
    print(f"  フック{r['hook']} 具体性{r['specificity']} 明確さ{r['clarity']} 共感{r['relatability']}"
          f" / 4軸計{total} / {r['binary']}")
    print()